In [4]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.evaluation import evaluate

SC = ROOT / "results" / "scores"
files = {
    "M1 IsolationForest": "m1_iforest_nftoniot_seed11.parquet",
    "M3 AE v1": "m3_autoencoder_nftoniot_seed11.parquet",
    "M3 AE v2": "m3v2_autoencoder_nftoniot_seed11.parquet",
		"M2 OneClassSVM": "m2_ocsvm_nftoniot_seed11.parquet",
}

rows = []
for name, f in files.items():
    df = pd.read_parquet(SC / f)
    val = df[df["split"] == "val"]
    for sp in ["tune", "test"]:
        d = df[df["split"] == sp]
        if d.empty:
            continue
        r = evaluate(d, val_df=val)          # thresholds come from benign val scores
        o = evaluate(d)                      # oracle: thresholds from the split's own benign scores
        rows.append({
            "model": name, "split": sp,
            "auroc": r["auroc"],
            "tpr@1%": r["tpr@fpr0.01"], "real_fpr@1%": r["real_fpr@0.01"],
            "tpr@5%": r["tpr@fpr0.05"], "macro_tpr@5%": r["macro_tpr@fpr0.05"],
            "real_fpr@5%": r["real_fpr@0.05"],
            "oracle_tpr@1%": o["tpr@fpr0.01"],
        })

table = pd.DataFrame(rows).round(3)
table

,model,split,auroc,tpr@1%,real_fpr@1%,tpr@5%,macro_tpr@5%,real_fpr@5%,oracle_tpr@1%
0,M1 IsolationForest,tune,0.806,0.005,0.011,0.070,0.387,0.102,0.004
1,M1 IsolationForest,test,0.520,0.100,0.073,0.563,0.620,0.570,0.002
2,M3 AE v1,tune,0.877,0.018,0.013,0.070,0.631,0.061,0.017
3,M3 AE v1,test,0.433,0.236,0.269,0.812,0.853,0.808,0.004
4,M3 AE v2,tune,0.953,0.007,0.013,0.931,0.927,0.084,0.004
5,M3 AE v2,test,0.395,0.242,0.536,0.923,0.941,0.859,0.003
6,M2 OneClassSVM,tune,0.868,0.004,0.015,0.046,0.531,0.051,0.002
7,M2 OneClassSVM,test,0.391,0.281,0.422,0.623,0.659,0.770,0.008


In [6]:
per = {}
for name, f in files.items():
    df = pd.read_parquet(SC / f)
    val = df[df["split"] == "val"]
    for sp in ["tune", "test"]:
        d = df[df["split"] == sp]
        if d.empty:
            continue
        per[(name, sp)] = evaluate(d, val_df=val)["per_attack_tpr@fpr0.01"]

pd.DataFrame(per).round(3)

M1 IsolationForest        M3 AE v1        M3 AE v2         \
                         tune   test     tune   test     tune   test   
ddos                    0.016  0.133    0.019  0.186    0.031  0.440   
dos                     0.000  0.000    0.008  0.151    0.063  0.163   
injection               0.105  0.083    0.311  0.267    0.234  0.173   
mitm                    0.000  0.040    0.000  0.274    0.000  0.321   
password                0.016  0.337    0.026  0.443    0.022  0.285   
scanning                0.004  0.026    0.014  0.108    0.003  0.102   
xss                     0.002  0.043    0.031  0.114    0.090  0.139   
backdoor                  NaN  0.004      NaN  0.997      NaN  0.998   
ransomware                NaN  0.100      NaN  0.792      NaN  0.780   

           M2 OneClassSVM         
                     tune   test  
ddos                0.012  0.653  
dos                 0.026  0.150  
injection           0.064  0.314  
mitm                0.000  0.230  
password            0.012  0.424  
scanning            0.002  0.010  
xss                 0.053  0.029  
backdoor              NaN  0.995  
ransomware            NaN  0.084

In [7]:
(ROOT / "results").mkdir(exist_ok=True)
table.to_csv(ROOT / "results" / "baseline_comparison.csv", index=False)
print("saved")

saved
